# Nutrition5k + TiMePReSt trên Kaggle T4 ×2
Kế hoạch: `n5k/PLAN.md`. Yêu cầu: Accelerator = **GPU T4 x2**, Internet = **On** (tải trọng số ImageNet), *Add Input* dataset **`gillesokhin/nutrition5k-dataset`** (có `dish_nutrition_values.csv`, `imagery/realsense_overhead/`).

**Lần chạy 1 (session này):** cell 1 → 5, rồi cell 6 đóng gói. Dán cho agent output của cell 3 (dòng `N5K_DATA_JSON`), cell 4 (bảng D0–D4 và dòng `N5K_CHECKS_JSON`) và cell 5 (bảng sweep + `N5K_SWEEP_JSON`). Agent chốt lr cho từng hệ rồi mới chạy dài (mục 7, lần chạy 2).

`/kaggle/working` bị xoá khi session kết thúc: chạy cell 6 và tải zip về trước khi tắt.

## 1. Repo + cài đặt + môi trường

In [ ]:
import os
REPO = '/kaggle/working/timeprest-reproduction'
if not os.path.exists(REPO):
    !git clone https://github.com/cotda/timeprest-reproduction.git {REPO}
%cd {REPO}
!git pull --ff-only
!git log --oneline -1
!pip install -q -e ".[test]"
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!python -c "import torch, torchvision; print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| cuda', torch.version.cuda, '| gpus', torch.cuda.device_count())"
!find /kaggle/input -maxdepth 4 -name dish_nutrition_values.csv

## 2. Unit test (CPU, ~10 phút; có thể bỏ qua nếu code không đổi so với lần trước)

In [ ]:
!python -m pytest -q timeprest/n5k/tests

## 3. Dữ liệu: giải mã ảnh RGB-D vào `/kaggle/working/n5k_cache` (một lần) và in báo cáo
Báo cáo gồm: số món mỗi split, món bị loại và lý do, số nguyên liệu K, tổng dinh dưỡng tính từ nguyên liệu có khớp tổng của món không.

In [ ]:
# tải trước trọng số ImageNet một lần (các tiến trình sau dùng cache, không tải song song)
!python -c "import torchvision; torchvision.models.resnet50(weights='IMAGENET1K_V2'); print('resnet50 IMAGENET1K_V2 ok')"
!python -m timeprest.n5k.data --config n5k/configs/n5k_quick.yaml

## 4. Check D0–D4 trên pipeline 2 GPU (PipeDream, TiMePReSt)
Phải ra `ALL N5K CHECKS PASS: True`. Ghi lại `peak_mem_mb_per_gpu`, `epoch_time_s`, `est_full_run_h` của D4.

In [ ]:
!torchrun --standalone --nproc_per_node=2 -m timeprest.n5k.checks --config n5k/configs/n5k_quick.yaml

## 5. Sweep lr (engine 1 tiến trình, 8 epoch/run, 2 run song song trên 2 GPU)
Hệ `single` (1 GPU, cũng dùng cho DDP), `pipedream`, `timeprest` × lr {0.01, 0.005, 0.002}. Chỉ chạy khi cell 4 PASS (trainer từ chối nếu chưa có file check). Log từng run ở `n5k/results/sweep/logs/`.

In [ ]:
!python -m timeprest.n5k.sweep --config n5k/configs/n5k_sweep.yaml --systems single pipedream timeprest --lrs 0.01 0.005 0.002 --gpus 0 1

## 6. Đóng gói kết quả (không gồm checkpoint) → tải `/kaggle/working/n5k_results.zip`

In [ ]:
!cd {REPO} && zip -qr /kaggle/working/n5k_results.zip n5k/results -x '*.pt' -x '*/work/*'
!ls -la /kaggle/working/n5k_results.zip

## 7. Lần chạy 2 — chạy dài (CHƯA chạy ở lần 1; agent cập nhật lr trong config trước)
Mỗi hệ 3 seed (0, 1, 2). Lệnh mẫu cho seed 0; seed khác thêm `--set seed=1 name=n5k_timeprest_s1` (tương tự cho các hệ khác). Bị ngắt thì chạy lại cùng lệnh với `--resume`.

In [ ]:
# TiMePReSt (N=2) và PipeDream trên pipeline 2 GPU
!torchrun --standalone --nproc_per_node=2 -m timeprest.n5k.dist_train --config n5k/configs/n5k_timeprest.yaml
!torchrun --standalone --nproc_per_node=2 -m timeprest.n5k.dist_train --config n5k/configs/n5k_pipedream.yaml

In [ ]:
# 1 GPU (mốc tham chiếu) và ablation N=4
!python -m timeprest.n5k.train --config n5k/configs/n5k_single.yaml
!torchrun --standalone --nproc_per_node=2 -m timeprest.n5k.dist_train --config n5k/configs/n5k_timeprest_n4.yaml